## Linguistic Tokenization

* **Segmentation:** Breaking text into smaller units for analysis
  * E.g. paragraphs, sentences, words ...
* **Tokenization:** Breaking text into word-like units (tokens)
  * Linguistic tokenization: Get linguistically meaningful analysis (e.g. word frequencies)


```
Extremely bad customer service

Do not go to this salon, especially if you have to get
your hair straightened. They did a very bad job with my
hair and were extremely rude when I went back to ask them
why it didn't work for my hair. Rude, insensitive,
discourteous people!!!!!
```
(*Text source: https://github.com/UniversalDependencies/UD_English-EWT*)


**Tokenized:**
```
Extremely bad customer service

Do not go to this salon , especially if you have to get
your hair straightened . They did a very bad job with my
hair and were extremely rude when I went back to ask them
why it did n't work for my hair . Rude , insensitive ,
discourteous people !!!!!
```

### Methods for linguistic tokenization

* **Naive method 1:** Split from whitespace characters

In [ ]:
text="""Extremely bad customer service

Do not go to this salon, especially if you have to get your hair straightened. \
They did a very bad job with my hair and were extremely rude when I went back to \
ask them why it didn't work for my hair. Rude, insensitive, discourteous people!!!!!"""

tokenized_text = text.split() # split(): Return a list of the words in the string, using whitespace as the delimiter string.

for w in tokenized_text:
    print(w)

* **Naive method 2:** Split from whitespace characters, take into account punctuation
* Regular expressions:
    * Define search patters
    * Find these patterns from raw text, or find-and-replace if needed
* Find all punctuation characters, and replace with whitespace+punctuation character
    * *book.* --> *book .*
    * *people!!!!!* --> *people !!!!!*
* Borderline cases:
    * How about clitics in English? [don't, can't, cannot?]
    * 2-(14-hydroxypentadecyl)-4-methyl-5-oxo-2,5-dihydrofuran-3-carboxylic acid ???
    * What will happen to numbers?
    * Usually it's not that important how exactly you do it, just be consistent!
      * consistent = always do it the same way
      * If you download two datasets which are already tokenized, the tokenization may differ and you need to be aware of it!

In [ ]:
import re

tokenized = re.sub(r'([.,!?]+)', r' \1', text) # replace . , ! ? with whitespace+character(s), '+' means one or more
tokenized = re.sub(r"(n't)", r" \1", tokenized) # clitics

print(tokenized) # Note: this is still string, apply simple whitespace splitting to get a list of tokens

* **Naive method 2** works quite well for English, Finnish, Swedish...
    * Approx. 97-99% correct on clean text
    * Many tokenizers are just a large number (in the hundreds) of regular expressions


* How about other languages, does it work for all?

.

.

.

.

.

.

.

.

**Nope! Why not?**

.

.

.

.

.

.

.

* All languages do not use whitespace or punctuation, or the meaning of those may be different.
* Chinese, Thai, Vietnamese

![tokenization.png](https://github.com/TurkuNLP/intro-to-nlp/blob/master/figs/tokenization.png?raw=1)

   
**Tokenization: State-of-the-art**
* State-of-the-art = The best existing method currently known
* Machine learning
    * Collect raw (untokenized) text for the language you are interested in, and manually tokenize it.
    * Train a classifier
    * The trained classifier can be used to tokenize new text

## Example of machine learned tokenizer (UDPipe)

In [ ]:
# Let's try to tokenize and sentence split a small dataset with UDPipe machine learned segmenter!
# Documentation: https://ufal.mff.cuni.cz/udpipe/users-manual
# Training data:
# Finnish (intro-to-nlp/Data/fi.segmenter.udpipe): https://github.com/UniversalDependencies/UD_Finnish-TDT v.2.2
# English (intro-to-nlp/Data/en.segmenter.udpipe): https://github.com/UniversalDependencies/UD_English-EWT v.2.2
# Swedish (intro-to-nlp/Data/sv.segmenter.udpipe): https://github.com/UniversalDependencies/UD_Swedish-Talbanken v.2.2

!wget -nc https://github.com/TurkuNLP/intro-to-nlp/raw/master/Data/en.segmenter.udpipe

!pip3 install ufal.udpipe

import ufal.udpipe as udpipe

model = udpipe.Model.load("en.segmenter.udpipe")
pipeline = udpipe.Pipeline(model,"tokenize","none","none","horizontal") # horizontal: returns one sentence per line, with words separated by a single space



In [ ]:
document="""
The North American X-15 is a hypersonic rocket-powered aircraft. It was operated by the United States
Air Force and the National Aeronautics and Space Administration as part of the X-plane series of
experimental aircraft. The X-15 set speed and altitude records in the 1960s, reaching
the edge of outer space and returning with valuable data used in aircraft and spacecraft
design. The X-15's highest speed, 4,520 miles per hour (7,274 km/h; 2,021 m/s),[1] was
achieved on 3 October 1967,[2] when William J. Knight flew at Mach 6.7 at an altitude of
102,100 feet (31,120 m), or 19.34 miles. This set the official world record for the highest
speed ever recorded by a crewed, powered aircraft, which remains unbroken.[3][4]

During the X-15 program, 12 pilots flew a combined 199 flights.[1] Of these,
8 pilots flew a combined 13 flights which met the Air Force spaceflight criterion
by exceeding the altitude of 50 miles (80 km), thus qualifying these pilots as being
astronauts; of those 13 flights, two (flown by the same civilian pilot) met the FAI
definition (100 kilometres (62 mi)) of outer space. The 5 Air Force pilots qualified
for military astronaut wings immediately, while the 3 civilian pilots were eventually
awarded NASA astronaut wings in 2005, 35 years after the last X-15 flight.[5][6]
"""

segmented_document = pipeline.process(document)

print(segmented_document)

## Calculate word frequencies

* How many running words are in the corpus (in total)?
* How many times each word appears in the corpus?
* How many unique words the corpus has?
    * vocabulary size

In [ ]:
!wget -nc https://github.com/TurkuNLP/intro-to-nlp/raw/master/Data/imdb_train.json

import json # JSON encoder and decoder: store python data structures (e.g. lists and dictionaries) as strings

with open("imdb_train.json", "rt", encoding="utf-8") as f:
    data = json.load(f)

print("Data type:", type(data))
print("First item type:", type(data[0]))
print("First item:", data[0])

In [ ]:
from collections import Counter
import tqdm

token_counter = Counter()
for doc in tqdm.tqdm(data[:1000]): # IMDB documents
    tokenized = pipeline.process(doc["text"])
    tokens = tokenized.split() # after segmenter, we can do whitespace splitting
    token_counter.update(tokens)

print("Number of tokens in total:", token_counter.total())
print("Most common tokens:")
for item in token_counter.most_common(20):
  print(item)
print("Vocabulary size:", len(token_counter))

## Subword tokenization for machine learning

Reduce vocabulary size by representing rare words with common subwords (playing → play + ing)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
text="""Extremely bad customer service

Do not go to this salon, especially if you have to get your hair straightened. \
They did a very bad job with my hair and were extremely rude when I went back to \
ask them why it didn't work for my hair. Rude, insensitive, discourteous people!!!!!"""

tokenized_text = tokenizer.tokenize(text)

for token in tokenized_text:
  print(token)

In [ ]:
# repeat word frequencies using subword tokenization

token_counter = Counter()
for doc in tqdm.tqdm(data[:1000]): # IMDB documents
    tokenized = tokenizer.tokenize(doc["text"])
    token_counter.update(tokenized)

print("Number of tokens in total:", token_counter.total())
print("Most common tokens:")
for item in token_counter.most_common(20):
  print(item)
print("Vocabulary size:", len(token_counter))